In [1]:
#分析训练集，得到非齐次生成乘客所用的λ_ij(t),λ_max,在Unhomogeneous RL train.ipynb使用
import os
import numpy as np
import pandas as pd

# ======================================================
# 基本配置
# ======================================================
ZONES = [237, 161, 236, 186, 162, 230, 142, 239, 163]
zone_to_idx = {z: i for i, z in enumerate(ZONES)}
N_ZONES = len(ZONES)

DATA_FILE = r"C:\Users\lenovo\Desktop\DQN2\data\yellow_tripdata_2025-10.parquet"

T_START = 17 * 3600
T_END   = 19 * 3600
T_TOTAL = 7200

BIN = 600
N_BIN = 12


DEGREE = 3

# ======================================================
# Step 1: 统计 10 分钟 OD 平均人数
# ======================================================
def compute_avg_10min_od_counts(file_path):
    total_counts = np.zeros((N_BIN, N_ZONES, N_ZONES))
    valid_days = set()
    
    # 读取整个10月数据
    df = pd.read_parquet(file_path)
    
    # 转换为日期时间
    df["pickup_dt"] = pd.to_datetime(df["tpep_pickup_datetime"])
    
    # 过滤条件：工作日
    df = df[
        (df["pickup_dt"].dt.month == 10) &
        (df["pickup_dt"].dt.weekday < 5)  # 周一至周五
    ]
    
    
    if df.empty:
        raise RuntimeError("No weekday data in October after removing holidays.")
    
    # 提取时间秒数
    sec = (
        df["pickup_dt"].dt.hour * 3600 +
        df["pickup_dt"].dt.minute * 60 +
        df["pickup_dt"].dt.second
    )
    
    # 筛选时间段：17:00-19:00
    df = df[(sec >= T_START) & (sec < T_END)].copy()
    if df.empty:
        raise RuntimeError("No data in the specified time window (17:00-19:00).")
    
    # 筛选指定区域
    df = df[
        df["PULocationID"].isin(ZONES) &
        df["DOLocationID"].isin(ZONES)
    ]
    if df.empty:
        raise RuntimeError("No data in the specified zones.")
    
    # 计算bin
    df["bin"] = ((sec.loc[df.index] - T_START) // BIN).astype(int)
    df["date"] = df["pickup_dt"].dt.date
    
    # 收集有效日期
    valid_days.update(df["date"].unique())
    
    # 按日期分组处理，避免内存溢出
    for date_str, date_group in df.groupby("date"):
        for _, r in date_group.iterrows():
            k = int(r["bin"])
            i = zone_to_idx[r["PULocationID"]]
            j = zone_to_idx[r["DOLocationID"]]
            total_counts[k, i, j] += 1
    
    day_count = len(valid_days)
    if day_count == 0:
        raise RuntimeError("No valid weekday data.")
    
    avg_counts = total_counts / day_count
    return avg_counts, day_count

# ======================================================
# Step 2: 多项式拟合 + 秒级 λ(t) + λ_max
# ======================================================
def fit_polynomial_and_lambda(avg_counts):
    # x = t / 600
    x = np.arange(N_BIN) + 0.5
    
    poly_params_10min = np.zeros((N_ZONES, N_ZONES, DEGREE + 1))  # f(x)的参数
    lambda_params_sec = np.zeros((N_ZONES, N_ZONES, DEGREE + 1))  # λ(t)的参数
    lambda_max = np.zeros((N_ZONES, N_ZONES))
    
    # 秒级时间网格（用于求最大值）
    t_fine = np.linspace(0, T_TOTAL, 4000)
    x_fine = t_fine / 600.0
    
    for i in range(N_ZONES):
        for j in range(N_ZONES):
            y = avg_counts[:, i, j]
            
            if y.mean() < 1e-6:
                continue
            
            # 1. 拟合 f(x)
            coeff_10min = np.polyfit(x, y, DEGREE)
            poly_params_10min[i, j] = coeff_10min
            
            # 2. 转换为 λ(t) 的参数
            a0, a1, a2, a3 = coeff_10min
            b0 = a0 / 600.0
            b1 = a1 / (600.0**2)
            b2 = a2 / (600.0**3)
            b3 = a3 / (600.0**4)
            
            lambda_params_sec[i, j] = [b0, b1, b2, b3]
            
            # 3. 计算 λ_max（用于thinning）
            lambda_fine = np.polyval(coeff_10min, x_fine) / 600.0
            lambda_fine = np.clip(lambda_fine, 0.0, None)
            lambda_max[i, j] = lambda_fine.max()
    
    return poly_params_10min, lambda_params_sec, lambda_max

# ======================================================
# 主流程
# ======================================================
if __name__ == "__main__":
    try:
        print(f"Analyzing file: {DATA_FILE}")
        avg_counts, n_days = compute_avg_10min_od_counts(DATA_FILE)
        print(f"Used weekdays: {n_days}")
        print("avg_counts shape:", avg_counts.shape)
        
        poly_params_10min, lambda_params_sec, lambda_max = fit_polynomial_and_lambda(avg_counts)
        
        # ==================================================
        # 输出结果
        # ==================================================
        print("\n=== λ_ij(t) parameters (per second) ===\n")
        for i in range(N_ZONES):
            for j in range(N_ZONES):
                b0, b1, b2, b3 = lambda_params_sec[i, j]
                if np.all(np.abs([b0, b1, b2, b3]) < 1e-10):
                    continue
                print(f"OD {ZONES[i]} -> {ZONES[j]} : λ(t) =")
                print(f"  {b0:.6e} + {b1:.6e}·t + {b2:.6e}·t² + {b3:.6e}·t³")
                print(f"  where t ∈ [0, 7200] seconds")
        
        print("\n=== λ_ij_max (per second) ===\n")
        for i in range(N_ZONES):
            for j in range(N_ZONES):
                if lambda_max[i, j] < 1e-10:
                    continue
                print(f"λ_max {ZONES[i]} -> {ZONES[j]} = {lambda_max[i, j]:.6e}")
                
    except Exception as e:
        print(f"Error occurred: {str(e)}")

Analyzing file: C:\Users\lenovo\Desktop\DQN2\data\yellow_tripdata_2025-10.parquet
Used weekdays: 23
avg_counts shape: (12, 9, 9)

=== λ_ij(t) parameters (per second) ===

OD 237 -> 237 : λ(t) =
  4.288519e-06 + -2.689440e-07·t + 4.249501e-09·t² + 6.457106e-11·t³
  where t ∈ [0, 7200] seconds
OD 237 -> 161 : λ(t) =
  -4.626345e-06 + 6.697208e-08·t + -1.070724e-10·t² + 4.115956e-11·t³
  where t ∈ [0, 7200] seconds
OD 237 -> 236 : λ(t) =
  -1.182392e-06 + -2.562051e-07·t + 5.005277e-09·t² + 9.565997e-11·t³
  where t ∈ [0, 7200] seconds
OD 237 -> 186 : λ(t) =
  4.682650e-06 + -1.109498e-07·t + 5.610211e-10·t² + 6.897683e-12·t³
  where t ∈ [0, 7200] seconds
OD 237 -> 162 : λ(t) =
  -6.024571e-06 + 6.068810e-08·t + 8.704002e-10·t² + 2.569841e-11·t³
  where t ∈ [0, 7200] seconds
OD 237 -> 230 : λ(t) =
  -1.873998e-05 + 4.862790e-07·t + -2.684213e-09·t² + 1.749939e-11·t³
  where t ∈ [0, 7200] seconds
OD 237 -> 142 : λ(t) =
  8.689646e-06 + -2.252779e-07·t + 1.816463e-09·t² + 3.252245e-11·t³
  